In [2]:
import sys
sys.executable

'c:\\Users\\anshuman\\Desktop\\toxic-comment-detection\\venv\\Scripts\\python.exe'

In [3]:
import pandas as pd 
import os 



In [4]:
df = pd.read_csv("train.csv")
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [5]:
text_column = "comment_text"
label_columns = ["toxic", "severe_toxic", "obscene", "threat","insult", "identity_hate"]

df[text_column].head()

0    Explanation\nWhy the edits made under my usern...
1    D'aww! He matches this background colour I'm s...
2    Hey man, I'm really not trying to edit war. It...
3    "\nMore\nI can't make any real suggestions on ...
4    You, sir, are my hero. Any chance you remember...
Name: comment_text, dtype: object

In [6]:
df.shape

(159571, 8)

In [7]:
df[label_columns].sum().sort_values(ascending=False)

toxic            15294
obscene           8449
insult            7877
severe_toxic      1595
identity_hate     1405
threat             478
dtype: int64

In [8]:
df["clean"] = (df[label_columns].sum(axis=1)==0)
df["clean"].value_counts()

clean
True     143346
False     16225
Name: count, dtype: int64

In [9]:
df[label_columns].sum(axis=1).value_counts().head()

0    143346
1      6360
3      4209
2      3480
4      1760
Name: count, dtype: int64

# Baseline model

In [10]:
X = df["comment_text"]
Y = df[label_columns]

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

model = OneVsRestClassifier(
    LogisticRegression(max_iter=1000)
)

model.fit(X_train_tfidf,Y_train)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...max_iter=1000)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=

In [14]:
y_pred = model.predict(X_test_tfidf)

In [15]:
from sklearn.metrics import classification_report

print(classification_report(Y_test, y_pred, target_names=label_columns))

               precision    recall  f1-score   support

        toxic       0.92      0.61      0.73      3056
 severe_toxic       0.59      0.21      0.30       321
      obscene       0.92      0.60      0.72      1715
       threat       0.47      0.09      0.16        74
       insult       0.85      0.51      0.64      1614
identity_hate       0.75      0.15      0.25       294

    micro avg       0.89      0.54      0.67      7074
    macro avg       0.75      0.36      0.47      7074
 weighted avg       0.88      0.54      0.66      7074
  samples avg       0.06      0.05      0.05      7074



c:\Users\anshuman\Desktop\toxic-comment-detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anshuman\Desktop\toxic-comment-detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\anshuman\Desktop\toxic-comment-detection\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(aver

here normal logistic regression and TF-IDF (baseline model) fails due to small represtation of some label in dataset and not considerring the context of word used. thus we use BERT so model can take decision based on the context of the comment also.
 

In [16]:
import pandas as pd
import numpy as np
import torch

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

c:\Users\anshuman\Desktop\toxic-comment-detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0513 14:25:06.349000 3520 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [17]:
df = pd.read_csv("train.csv")

df.head()


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [18]:
labels = df.columns[2:]

df['labels'] = df[labels].values.tolist()



In [19]:
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

In [20]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

c:\Users\anshuman\Desktop\toxic-comment-detection\venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [21]:
train_encodings = tokenizer(
    train_df["comment_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    val_df["comment_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

In [22]:
class ToxicDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx]).float()

        return item

    def __len__(self):
        return len(self.labels)

In [23]:
train_dataset = ToxicDataset(train_encodings, train_df["labels"].tolist())
val_dataset = ToxicDataset(val_encodings, val_df["labels"].tolist())



In [24]:
print(len(train_dataset))
print(len(val_dataset))

143613
15958


In [25]:
from torch.utils.data import Subset

In [26]:
train_dataset = Subset(train_dataset, range(20000))
val_dataset = Subset(val_dataset, range(5000))

In [27]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=6,
    problem_type="multi_label_classification"
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [28]:
import sys
print(sys.executable)

c:\Users\anshuman\Desktop\toxic-comment-detection\venv\Scripts\python.exe


In [29]:
!pip install accelerate


[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
import accelerate
print(accelerate.__version__)

0.27.2


In [31]:
import accelerate
import transformers
import torch

print("Accelerate:", accelerate.__version__)
print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)

Accelerate: 0.27.2
Transformers: 4.38.2
Torch: 2.9.1+cpu


In [32]:
from transformers import TrainingArguments

In [33]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100
)

In [34]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=6
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [35]:
print(type(train_dataset))

print("df" in globals())
print("train_dataset" in globals())
print("val_dataset" in globals())
print("model" in globals())

<class 'torch.utils.data.dataset.Subset'>
True
True
True
True


In [36]:
!pip install transformers==4.38.2
!pip install accelerate==0.27.2


[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
from transformers import Trainer

In [38]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset
)

In [41]:
print(len(train_dataset))
print(len(val_dataset))

20000
5000


In [43]:
print(training_args.num_train_epochs)

1


In [45]:
trainer.train()

  4%|▍         | 100/2500 [03:22<1:45:07,  2.63s/it]

{'loss': 0.1152, 'grad_norm': 0.41188153624534607, 'learning_rate': 1.9200000000000003e-05, 'epoch': 0.04}


  8%|▊         | 200/2500 [06:36<1:07:46,  1.77s/it]

{'loss': 0.0685, 'grad_norm': 0.8222015500068665, 'learning_rate': 1.8400000000000003e-05, 'epoch': 0.08}


 12%|█▏        | 300/2500 [09:30<1:05:25,  1.78s/it]

...

                                                     
100%|██████████| 2500/2500 [1:17:23<00:00,  1.57s/it]Checkpoint destination directory ./results\checkpoint-2500 already exists and is non-empty. Saving will proceed but saved results may be invalid.


{'eval_loss': 0.04354006424546242, 'eval_runtime': 239.7274, 'eval_samples_per_second': 20.857, 'eval_steps_per_second': 2.607, 'epoch': 1.0}


100%|██████████| 2500/2500 [1:17:24<00:00,  1.86s/it]

{'train_runtime': 4644.6814, 'train_samples_per_second': 4.306, 'train_steps_per_second': 0.538, 'train_loss': 0.05778704891204834, 'epoch': 1.0}


TrainOutput(global_step=2500, training_loss=0.05778704891204834, metrics={'train_runtime': 4644.6814, 'train_samples_per_second': 4.306, 'train_steps_per_second': 0.538, 'train_loss': 0.05778704891204834, 'epoch': 1.0})

In [46]:
predictions = trainer.predict(val_dataset)

c:\Users\anshuman\Desktop\toxic-comment-detection\venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
100%|██████████| 625/625 [04:15<00:00,  2.45it/s]


In [47]:
import torch

logits = predictions.predictions
probs = torch.sigmoid(torch.tensor(logits))

In [48]:
y_pred = (probs > 0.5).int().numpy()

In [49]:
y_true = val_df[labels].values

In [51]:
preds = trainer.predict(val_dataset)

100%|██████████| 625/625 [03:53<00:00,  2.67it/s]


In [52]:
y_pred = preds.predictions
y_true = preds.label_ids

In [53]:
from scipy.special import expit
import numpy as np

y_pred = expit(y_pred)
y_pred = (y_pred > 0.5).astype(int)

In [54]:
from sklearn.metrics import f1_score

f1_micro = f1_score(y_true, y_pred, average="micro")
f1_macro = f1_score(y_true, y_pred, average="macro")

print("Micro F1:", f1_micro)
print("Macro F1:", f1_macro)

Micro F1: 0.751008064516129
Macro F1: 0.39950426778793185


In [55]:
trainer.save_model("toxic_model")
tokenizer.save_pretrained("toxic_model")

('toxic_model\\tokenizer_config.json',
 'toxic_model\\special_tokens_map.json',
 'toxic_model\\vocab.txt',
 'toxic_model\\added_tokens.json',
 'toxic_model\\tokenizer.json')

In [56]:
import torch
from scipy.special import expit

labels = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

def predict_toxicity(text):
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = expit(outputs.logits.numpy())[0]

    predictions = (probs > 0.5).astype(int)

    for label, pred in zip(labels, predictions):
        print(f"{label}: {pred}")

In [57]:
predict_toxicity("You are an idiot and I hate you")

toxic: 1
severe_toxic: 0
obscene: 1
threat: 0
insult: 1
identity_hate: 0


In [58]:
predict_toxicity("You are stupid")
predict_toxicity("I will kill you")
predict_toxicity("Have a nice day")

toxic: 1
severe_toxic: 0
obscene: 1
threat: 0
insult: 1
identity_hate: 0
toxic: 1
severe_toxic: 0
obscene: 1
threat: 0
insult: 1
identity_hate: 0
toxic: 0
severe_toxic: 0
obscene: 0
threat: 0
insult: 0
identity_hate: 0
